# Lean-18 : A* et l'optimalité sous heuristique admissible — visite formelle de `astar_lean`

**Navigation** : [<< Lean-17b Knots Invariants](Lean-17-Knots-b-Invariants-Companion.ipynb) | [Index](README.md)

**Kernel** : Python 3 (extraits Mathlib/astar_lean exhibés via `subprocess` -> WSL `lean`)

**Compagnon** : lake [`astar_lean`](../../Search/astar_lean/) (série Search, issue [#4048](https://github.com/jsboige/CoursIA/issues/4048), roadmap [#4038](https://github.com/jsboige/CoursIA/issues/4038)).

---

> *A\* est optimal si l'heuristique n'est pas optimiste — i.e. elle ne surestime
> jamais le coût restant. Cette « admissibilité » est la condition exacte que
> formalise `astar_lean`, dont on visite ici les preuves (0 `sorry`).*


## Introduction : pourquoi formaliser l'optimalité de A\* ?

L'algorithme **A\*** (Hart, Nilsson & Raphael, 1968) déploie le nœud de fonction
d'évaluation `f(n) = g(n) + h(n)` minimale — `g` coût déjà parcouru, `h`
heuristique estimant le coût restant. La garantie d'**optimalité** (A\* renvoie un
chemin de coût minimal) repose sur une hypothèse précise : l'heuristique doit être
**admissible** (`h ≤ h*`, le vrai coût optimal restant).

Le lake `astar_lean` prouve **formellement** (0 `sorry`) le mécanisme exact de cette
optimalité : la **borne en `f`** qui, sous heuristique admissible, garantit qu'A\*
ne dépasse jamais la frontière du coût optimal. Il établit aussi la chaîne
**consistance ⟹ admissibilité** (par téléscopage) et la **connexion à Dijkstra**
(heuristique nulle = recherche à coût uniforme).

Cette visite exhibe les définitions et théorèmes clés du lake via leurs sources,
vérifie qu'ils sont bien à `0 sorry`, et les fait manipuler dans des exercices
exécutés par le kernel Lean (via `lake env lean`).

**Plan** : (1) Graphes pondérés et coût de chemin → (2) Admissibilité et
consistance → (3) Théorème phare d'optimalité → (4) Téléscopage consistance⟹admissible →
(5) Connexion à Dijkstra → (6) Exercices.


In [1]:
import subprocess
import textwrap
import re
import os
import shutil
import tempfile
from pathlib import Path

# --- Resolution du chemin du lake astar_lean ---
# Doit fonctionner en interactif (CWD = racine repo) et sous Papermill.

def find_astar_lean_project():
    """Localise le repertoire du lake astar_lean (contient lakefile.lean).

    Robuste a plusieurs contextes d'execution : interactif VSCode (CWD = dir du
    notebook, __vsc_ipynb_file__ defini), papermill natif Windows, et papermill
    dans WSL (CWD = home de login, hors repo). Strategie : on collecte plusieurs
    racines candidates et on cherche le lake comme enfant direct d'un ancetre
    (convention grothendieck_lean) OU comme <ancetre>/Search/astar_lean
    (convention cross-branche, car le notebook est dans SymbolicAI/Lean/ mais le
    lake dans Search/)."""
    starts = [Path.cwd()]
    nb_file = os.environ.get('NB_FILE') or globals().get('__vsc_ipynb_file__')
    if nb_file:
        starts.append(Path(nb_file).parent)
    # Ancres explicites (papermill WSL : CWD hors repo). Formes Windows + WSL.
    starts.extend([Path('C:/dev/CoursIA'), Path('/mnt/c/dev/CoursIA')])

    seen = set()
    for start in starts:
        try:
            current = Path(start).resolve()
        except Exception:
            continue
        for _ in range(16):
            if current in seen:
                break
            seen.add(current)
            cands = (
                current / 'astar_lean',
                current / 'Search' / 'astar_lean',
                current / 'MyIA.AI.Notebooks' / 'Search' / 'astar_lean',
            )
            for cand in cands:
                if cand.exists() and (cand / 'lakefile.lean').exists():
                    return cand.resolve()
            if current == current.parent:
                break
            current = current.parent
    raise FileNotFoundError("astar_lean/ introuvable -- verifier le working directory")

def win_to_wsl(win_path: Path) -> str:
    """Convertit un chemin Windows en chemin WSL (/mnt/<drive>/...)."""
    p = win_path.resolve()
    drive_letter = p.drive
    if not drive_letter or len(drive_letter) < 2:
        s = str(p)
        return s if s.startswith('/mnt/') else s
    drive = drive_letter[0].lower()
    return f'/mnt/{drive}{p.as_posix()[2:]}'

WIN_LEAN_PROJECT = find_astar_lean_project()
LEAN_PROJECT = win_to_wsl(WIN_LEAN_PROJECT)
USE_NATIVE_LEAN = shutil.which('lake') is not None and os.name != 'nt'
print(f"Lake astar_lean detecte   : {WIN_LEAN_PROJECT.name} (sous {WIN_LEAN_PROJECT.parent.name}/)")
print(f"Chemin WSL                 : (normalise, non affiche pour #3436)")
print(f"Lean natif (hors WSL)     : {USE_NATIVE_LEAN}")

def wsl(cmd, timeout=60):
    """Execute une commande bash dans WSL Ubuntu. Capture stdout/stderr via
    fichiers temporaires pour eviter la race CPython _readerthread sur Windows."""
    full = ['wsl', '-d', 'Ubuntu', '--', 'bash', '-lc', cmd]
    out_f = tempfile.NamedTemporaryFile('wb', delete=False, suffix='.out')
    err_f = tempfile.NamedTemporaryFile('wb', delete=False, suffix='.err')
    out_path, err_path = out_f.name, err_f.name
    out_f.close(); err_f.close()
    try:
        with open(out_path, 'wb') as o, open(err_path, 'wb') as e:
            r = subprocess.run(full, stdout=o, stderr=e, timeout=timeout)
        out = Path(out_path).read_text(encoding='utf-8', errors='replace')
        err = Path(err_path).read_text(encoding='utf-8', errors='replace')
        return r.returncode, out, err
    except FileNotFoundError:
        return 127, '', 'WSL executable not found'
    except subprocess.TimeoutExpired:
        return -1, '', f'TIMEOUT after {timeout}s'
    finally:
        for p in (out_path, err_path):
            try: Path(p).unlink()
            except OSError: pass

# --- Lecture des fichiers .lean du lake ---
def read_lean_module(module_name):
    """Lit un fichier source .lean du lake astar_lean.
    module_name ex: 'Optimality' -> Astar/Optimality.lean"""
    p = WIN_LEAN_PROJECT / 'Astar' / f'{module_name}.lean'
    if not p.exists():
        return f'[FICHIER INTROUVABLE] {p}'
    return p.read_text(encoding='utf-8')

def display_lean_module(module_name, max_lines=None, highlight=None):
    """Affiche un fichier .lean avec numeros de ligne.
    highlight: liste de numeros de ligne (1-indexes) marques '>>>'."""
    content = read_lean_module(module_name)
    if content.startswith('[FICHIER INTROUVABLE]'):
        print(content); return
    lines = content.splitlines()
    if max_lines: lines = lines[:max_lines]
    highlight = set(highlight or [])
    print(f'--- Astar/{module_name}.lean ---')
    for i, line in enumerate(lines, 1):
        marker = ' >>>' if i in highlight else '    '
        print(f'{marker} {i:>3d} | {line}')
    total = len(content.splitlines())
    if max_lines and total > max_lines:
        print(f'    ... ({total - max_lines} lignes restantes sur {total} total)')
    print(f'--- fin ({total} lignes) ---')

def run_lake_build(targets='Astar', timeout=1500):
    """Construit le lake astar_lean."""
    if USE_NATIVE_LEAN:
        try:
            r = subprocess.run(['lake', 'build', targets], cwd=WIN_LEAN_PROJECT,
                               capture_output=True, text=True, timeout=timeout)
            return r.returncode, r.stdout, r.stderr
        except subprocess.TimeoutExpired:
            return -1, '', f'TIMEOUT after {timeout}s'
    # Capture du VRAI exit code de lake (pas celui de `tail` qui masque les
    # echecs -- incident : exit=0 trompeur alors que lake avortait sur le
    # checkout mathlib). On ecrit la sortie dans un fichier, on recupere $?,
    # puis on sort avec ce code.
    return wsl(
        f'source ~/.elan/env 2>/dev/null; cd {LEAN_PROJECT}; '
        f'lake build {targets} > /tmp/lean18_build.out 2>&1; rc=$?; '
        f'tail -25 /tmp/lean18_build.out; exit $rc',
        timeout=timeout)

def run_lean(snippet, timeout_s=300):
    """Execute un snippet Lean contre le lake astar_lean via `lake env lean`.
    Le snippet est ecrit dans un fichier temporaire et execute dans l'env Lake."""
    snippet = textwrap.dedent(snippet).strip() + '\n'
    if USE_NATIVE_LEAN:
        with tempfile.NamedTemporaryFile('w', suffix='.lean', delete=False, encoding='utf-8') as tmp:
            tmp.write(snippet); tmp_path = tmp.name
        try:
            r = subprocess.run(['lake', 'env', 'lean', tmp_path], cwd=WIN_LEAN_PROJECT,
                               capture_output=True, text=True, timeout=timeout_s)
            return (r.stdout or '') + (r.stderr or '')
        except subprocess.TimeoutExpired:
            return f'TIMEOUT after {timeout_s}s'
        finally:
            try: Path(tmp_path).unlink()
            except OSError: pass
    # WSL : ecrire le snippet puis l'executer
    import uuid
    tag = uuid.uuid4().hex[:8]
    write_cmd = f"cat > /tmp/lean18_snippet_{tag}.lean << 'LEAN_EOF'\n{snippet}LEAN_EOF"
    exec_cmd = f"source ~/.elan/env 2>/dev/null; cd {LEAN_PROJECT} && lake env lean /tmp/lean18_snippet_{tag}.lean 2>&1"
    # Saut de ligne (pas '&&') : sinon '&& exec_cmd' se colle au delimiteur
    # LEAN_EOF et bash ne ferme jamais le heredoc (sortie vide). Per Lean-15.
    rc, out, err = wsl(write_cmd + chr(10) + exec_cmd, timeout=timeout_s)
    return out + err


Lake astar_lean detecte   : astar_lean (sous Search/)
Chemin WSL                 : (normalise, non affiche pour #3436)
Lean natif (hors WSL)     : False


In [2]:
# Verification : le lake astar_lean est a 0 sorry et construit
import re
SORRY_RE = re.compile(r'^\s*sorry\s*$|:=\s*by\s*sorry|:=\s*sorry\s*$|\bexact\s+sorry\b|^\s*[\u00b7-]\s*sorry\b')
ASTAR_MODULES = ['Graph', 'Heuristic', 'Optimality', 'Consistency']
total_sorry = 0
for mod in ASTAR_MODULES:
    src = read_lean_module(mod)
    n = len(SORRY_RE.findall(src))
    total_sorry += n
    print(f"  Astar/{mod}.lean : {n} sorry")
print(f"\nTotal sorry (4 modules) : {total_sorry}")
print(f"astar_lean est FORMELLEMENT CERTIFIE : {'OUI' if total_sorry == 0 else 'NON'}")


  Astar/Graph.lean : 0 sorry
  Astar/Heuristic.lean : 0 sorry
  Astar/Optimality.lean : 0 sorry
  Astar/Consistency.lean : 0 sorry

Total sorry (4 modules) : 0
astar_lean est FORMELLEMENT CERTIFIE : OUI


In [3]:
# Build du lake (confirme que les preuves compilent reellement).
# run_lake_build capture le VRAI exit code de lake (pas celui de `tail`, qui
# masquerait un echec). Astuce : le build complet d'Astar importe Mathlib et
# requiert les oleans Mathlib ; s'ils manquent, on le signale honnetement et on
# imprime la commande manuelle (convention Lean-15).
rc, out, err = run_lake_build('Astar', timeout=600)
if rc == 0:
    print(f"lake build Astar -> exit={rc} : SUCCESS, preuves compilees, 0 sorry verifie par build.")
    if out.strip():
        print(out.strip()[-500:])
else:
    print(f"lake build Astar -> exit={rc} : build non complete sur cette machine")
    print("(oles Mathlib manquants : `lake exe cache get` ne repond pas ici).")
    print()
    print("La verification statique ci-dessus (regex sur les sources) confirme deja 0 sorry.")
    print("Pour valider par compilation, lancer sur une machine avec l'env Lean/WSL complet :")
    print(f'  wsl -d Ubuntu -- bash -lc "cd <astar_lean> && lake exe cache get && lake build Astar"')


lake build Astar -> exit=0 : SUCCESS, preuves compilees, 0 sorry verifie par build.
Build completed successfully (8497 jobs).


## 1. Graphes pondérés et coût d'un chemin

Le modèle abstrait de `astar_lean` pose un **graphe orienté pondéré** à arêtes
non-négatives (`NNReal ≡ ℝ≥0`). La non-négativité des poids est exactement
l'hypothèse requise pour l'optimalité de A\*. Un **chemin** est une liste de
sommets, et son **coût** est la somme (récursive) des poids des arcs consécutifs.


In [4]:
# Source : WeightedGraph, pathCost, PathFrom
display_lean_module('Graph', highlight=[20, 33, 50])


--- Astar/Graph.lean ---
       1 | import Mathlib
       2 | 
       3 | /-!
       4 | # Astar.Graph — graphes pondérés, chemins, coût d'un chemin
       5 | 
       6 | Modèle abstrait pour la preuve d'optimalité de A* (issue #4048). Un graphe orienté
       7 | pondéré à arêtes non-négatives (`NNReal` ≡ ℝ≥0), des chemins vus comme des listes de
       8 | sommets, et le coût d'un chemin comme somme des poids des arcs consécutifs.
       9 | -/
      10 | 
      11 | namespace Astar
      12 | 
      13 | /- Sommets abstraits : `V` est le type des sommets, paramètre du modèle. -/
      14 | variable {V : Type*}
      15 | 
      16 | /-- Graphe orienté pondéré : `edge a b` est le coût non-négatif de l'arc `a → b`.
      17 |     La valeur `0` signifie « pas d'arc » (ou boucle triviale). L'hypothèse de
      18 |     non-négativité (`NNReal`) est exactement l'hypothèse requise pour l'optimalité
      19 |     de A*. -/
 >>>  20 | structure WeightedGraph (V : Type*) where
      21 |  

### Lecture : `WeightedGraph`, `pathCost`, `PathFrom`

| Symbole Lean | Lecture |
|--------------|---------|
| `WeightedGraph V` | Graphe porté par une fonction d'arc `edge : V → V → ℝ≥0` (0 = pas d'arc) |
| `pathCost G [v₀, v₁, …]` | Somme des poids des arcs : `edge v₀ v₁ + edge v₁ v₂ + …` |
| `pathCost [] = 0`, `pathCost [v] = 0` | Un chemin vide ou à un seul sommet a un coût nul (aucun arc) |
| `PathFrom start goal p` | Le chemin `p` va bien de `start` à `goal` (non-vide, bonne tête, bonne fin) |

Le coût `pathCost` est défini par filtrage de motif sur la liste — les trois
lemmas `pathCost_nil` / `pathCost_singleton` / `pathCost_cons_cons` (prouvés par
`rfl`) en figent le calcul. C'est le seul ingrédient combinatoire nécessaire : la
preuve d'optimalité ne dépend pas d'une structure de donnée de file, seulement du
coût additif des chemins.


## 2. Heuristique admissible et consistante

Deux prédicats centraux structurent toute la théorie :

- **`Admissible h hStar`** : `h n ≤ hStar n` pour tout sommet `n` — l'heuristique
  ne **surestime** jamais le vrai coût optimal restant `hStar`. C'est l'hypothèse
  **globale** d'optimalité.
- **`Consistent G h`** : `h n ≤ edge n n' + h n'` pour tout arc — la consistance
  (ou monotonie) est une condition **locale** (relaxation de l'équation de Bellman).

Le lake prouve leurs propriétés de base : **monotonie** (`admissible_mono`),
**fermeture par minimum** (`admissible_min`, base de la combinaison
d'heuristiques), **l'heuristique parfaite est admissible** (`hStar_admissible`), et
surtout la **connexion à Dijkstra** : l'heuristique nulle `h ≡ 0` est admissible
ET consistante — A\* avec heuristique nulle se réduit à la recherche à coût uniforme.


In [5]:
# Source : Admissible, Consistent + lemmas compagnons
display_lean_module('Heuristic', highlight=[30, 37, 65, 71, 76])


--- Astar/Heuristic.lean ---
       1 | import Mathlib
       2 | import Astar.Graph
       3 | 
       4 | /-!
       5 | # Astar.Heuristic — admissibilité et consistance
       6 | 
       7 | Définitions centrales de A*. Soit `hStar : V → NNReal` le « vrai coût optimal
       8 | restant » (le coût minimal pour atteindre le but depuis chaque sommet). Une
       9 | heuristique `h : V → NNReal` est :
      10 | 
      11 | - **admissible** si `h n ≤ hStar n` pour tout sommet `n` : elle ne surestime
      12 |   jamais le coût optimal restant ;
      13 | - **consistante** (ou monotone) si `h n ≤ edge n n' + h n'` pour tout arc
      14 |   `n → n'` : c'est la « relaxation » de l'équation de Bellman (programmation
      15 |   dynamique). La consistance implique l'admissibilité (voir `Optimality.lean`).
      16 | 
      17 | `hStar` reste ici abstrait ; sa propriété caractéristique de borne inférieure
      18 | sur le coût des chemins menant au but est énoncée dans `Optimality.lean`

### Lecture : pourquoi `h ≡ 0` est admissible (connexion à Dijkstra)

Les lemmas `zero_admissible` et `zero_consistent` (prouvés en une ligne chacun,
`zero_le` en `ℝ≥0`) sont l'ancrage pédagogique : **Dijkstra est le cas particulier
d'A\* avec une heuristique nulle**. Toute la machinerie d'optimalité que l'on prouve
pour A\* admissible s'applique donc *a fortiori* à Dijkstra.

| Symbole | Idée |
|---------|------|
| `admissible_mono` | Une heuristique majorée par une admissible reste admissible (pour « raboter ») |
| `admissible_min` | Le `min` de deux admissibles est admissible (combinaison d'heuristiques) |
| `hStar_admissible` | Le vrai coût optimal est lui-même admissible (borne supérieure) |
| `zero_admissible` / `zero_consistent` | Heuristique nulle admissible + consistante ⇒ Dijkstra |


## 3. Théorème phare : admissible ⟹ optimal (borne en `f`)

C'est le cœur mathématique de l'optimalité de A\*. On garde `hStar` abstrait — un
« vrai coût optimal restant » satisfaisant la propriété de **borne inférieure**
`IsTrueRemainingCost` (pour un graphe fini, `hStar` = minimum des coûts des chemins
simples vers le but). Le théorème phare `admissible_implies_optimal` énonce la
**borne en `f`** : sous heuristique admissible, pour tout nœud d'un chemin allant au
but, `h(nœud)` ne dépasse jamais le coût du suffixe restant. Combinée à
`f = g + h` et au fait que `g + suffixCost = pathCost`, cette borne donne
`f(nœud) ≤ coût optimal` le long du chemin optimal — A\* (qui déploie le `f`
minimal) ne dépasse jamais la frontière du coût optimal.


In [6]:
# Source : IsTrueRemainingCost + theoreme phare admissible_implies_optimal
display_lean_module('Optimality', highlight=[33, 61, 76, 85])


--- Astar/Optimality.lean ---
       1 | import Mathlib
       2 | import Astar.Graph
       3 | import Astar.Heuristic
       4 | 
       5 | /-!
       6 | # Astar.Optimality — admissible ⇒ optimal (lemme abstrait d'optimalité)
       7 | 
       8 | Théorème phare de la série (issue #4048, registre #3801 — prong B « problème
       9 | non-trivial »). On prouve la **borne en `f`** qui est le mécanisme exact d'optimalité
      10 | de A* : sous une heuristique admissible, `h(start)` ne dépasse jamais le coût d'un
      11 | chemin réalisé jusqu'au but. Appliquée à chaque suffixe du chemin optimal, cette borne
      12 | donne `f(nœud) = g(nœud) + h(nœud) ≤ coût optimal` en tout point du chemin optimal —
      13 | donc A* (qui déploie le nœud de `f` minimal) ne dépasse jamais la frontière du coût
      14 | optimal et renvoie un chemin de coût optimal (Hart, Nilsson & Raphael, 1968).
      15 | 
      16 | On prouve la **forme abstraite** (sans modéliser la file de priorité complète)

### Lecture : le mécanisme exact d'optimalité

Le théorème `admissible_implies_optimal` (Optimality.lean, mis en évidence ci-dessus)
se prouve en **deux pas** :

1. **`suffix_pathFrom`** (lemme auxiliaire) : un suffixe d'un chemin allant au but
   va encore au but — pure manipulation de listes (`List.head?_drop`, `List.getLast?_drop`).
2. **Composition des bornes** : `h(nœud) ≤ hStar(nœud)` (admissibilité) puis
   `hStar(nœud) ≤ pathCost(suffixe)` (`IsTrueRemainingCost` appliqué au suffixe via
   le lemme 1) — donc `h(nœud) ≤ pathCost(suffixe)` par `le_trans`.

C'est la **borne en `f` en chaque nœud** : `f(nœud) = g(nœud) + h(nœud) ≤
g(nœud) + pathCost(suffixe) = pathCost(chemin complet)`. Sur le chemin optimal,
cela vaut le coût optimal — d'où l'optimalité de A\*. (Hart, Nilsson & Raphael, 1968.)

> **Cadrage honnête (per #4048)** : la forme prouvée est **abstraite** — `hStar`
> est une borne inférieure sur les coûts de chemins, pas nécessairement un coût
> réalisé. C'est délibéré : on prouve le mécanisme d'optimalité sans modéliser la
> file de priorité complète. La réalisabilité de `hStar` (graphe fini ⇒ minimum
> atteint) est laissée abstraite, plus propre pédagogiquement.


## 4. Consistance ⟹ admissible (le téléscopage)

La **consistance** est locale (par arc) ; l'**admissibilité** est globale. Le pont
est un **téléscopage** : le long des arcs d'un chemin `start → … → goal`, la
consistance se compose en

```
h(start) ≤ edge(v₀,v₁) + h(v₁)
         ≤ edge(v₀,v₁) + edge(v₁,v₂) + h(v₂)
         ≤ …
         ≤ pathCost(p) + h(goal)
```

Sous `h(goal) = 0`, il vient **`h(start) ≤ pathCost(p)`** — la même borne globale
que l'admissibilité, atteinte sans hypothèse sur `hStar`. Le lake prouve aussi que
la consistance rend `f = g + h` **monotone** le long des expansions
(`consistent_implies_f_monotone`) — d'où la **non-ré-expansion** des nœuds.


In [7]:
# Source : telecopage consistent_implies_path_bound + monotonicite de f
display_lean_module('Consistency', highlight=[55, 95, 123])


--- Astar/Consistency.lean ---
       1 | import Mathlib
       2 | import Astar.Graph
       3 | import Astar.Heuristic
       4 | import Astar.Optimality
       5 | 
       6 | /-!
       7 | # Astar.Consistency — consistance ⟹ admissibilité (téléscopage)
       8 | 
       9 | Issue #4048, théorème cible `consistent_implies_admissible`. La **consistance**
      10 | (monotonie par arc : `h n ≤ edge n n' + h n'`) est une condition **locale** ;
      11 | l'**admissibilité** (`h n ≤ hStar n`) est une condition **globale**. Le pont entre les
      12 | deux est un **téléscopage** : le long des arcs d'un chemin `start = v₀ → v₁ → … → vₖ =
      13 | goal`, la consistance se compose en
      14 | 
      15 | ```
      16 | h(start) ≤ edge(v₀,v₁) + h(v₁)
      17 |          ≤ edge(v₀,v₁) + edge(v₁,v₂) + h(v₂)
      18 |          ≤ …
      19 |          ≤ pathCost(p) + h(goal).
      20 | ```
      21 | 
      22 | Sous l'hypothèse naturelle `h(goal) = 0` (l'heuristique est nulle au but), 

### Lecture : du local au global, et l'efficacité

Le théorème `consistent_implies_path_bound` (mis en évidence) prouve le
téléscopage par **récurrence sur la queue de la liste**, avec `linarith` pour
combiner la consistance à l'arc courant et l'hypothèse de récurrence. C'est le
résultat substantiel pleinement prouvé : la condition **locale** atteint
gratuitement la borne **globale**.

| Théorème | Conclusion |
|----------|------------|
| `consistent_implies_path_bound` | Consistance + `h(goal)=0` ⇒ `h(start) ≤ pathCost(p)` pour tout chemin `p` |
| `consistent_implies_admissible_bound` | Corollaire : la même borne que l'admissibilité, sans hypothèse sur `hStar` |
| `consistent_implies_f_monotone` | Consistance ⇒ `f = g + h` croissante ⇒ aucun nœud ré-expansé (efficacité) |

**Leçon** : la consistance renforce l'admissibilité — elle garantit non seulement
l'optimalité (comme l'admissibilité) mais aussi l'**efficacité** (pas de
ré-expansion). C'est pourquoi les heuristiques consistantes (ex. distance de
Manhattan, distance euclidienne) sont préférées en pratique.


## 5. La chaîne causale complète

Les quatre modules composent une chaîne unique, du local au global, qui culmine
dans l'optimalité de A\* :

```
consistance (locale, par arc)
   └─[téléscopage, Consistency.lean]─▶ h(start) ≤ pathCost(p)  (borne globale)
                                            │
                                            ├─▶ borne en f = g + h ≤ coût optimal  ⟹ A* OPTIMAL
                                            └─▶ f monotone  ⟹ A* EFFICACE (pas de ré-expansion)

admissibilité (globale, h ≤ hStar)
   └─[Optimality.lean]─▶ borne en f en chaque nœud  ⟹ A* OPTIMAL

cas particulier h ≡ 0
   └─▶ Dijkstra (recherche à coût uniforme)
```

Tout cela est **formellement prouvé à 0 `sorry`** dans `astar_lean` — la garantie
que l'optimalité de A\* n'est pas un argument de manuel mais un théorème vérifié
mécaniquement.


## 6. Exemple guidé et exercices

On manipule les structures de `astar_lean`. D'abord un **exemple guidé résolu**
(les signatures réelles, lues directement depuis les sources du lake), puis
**trois exercices** à compléter : chaque squelette est un fragment Lean contenant
un `sorry`/blanc (`# TODO étudiant`) à remplir. Pour vérifier vos solutions,
ouvrez le lake dans un éditeur Lean (comme VS Code + l'extension `lean4`) ou
lancez `lake env lean <fichier>` après un `lake build`. Les exercices ne sont **pas**
exécutés tant que vous ne les avez pas complétés — le notebook reste exécutable
de bout en bout.


In [8]:
# Exemple guide (RESOLU) : signatures des theoremes phares.
# On extrait les DECLARATIONS reelles depuis les sources du lake (lecture
# directe, independante de l'env) plutot que via `lake env lean` (qui requiert
# les oleans Mathlib, absents sur cette machine -- convention Lean-15).

import re
def extract_signatures(mod, names):
    """Extrait les lignes de declaration (theorem/def/lemma) pour `names`."""
    src = read_lean_module(mod)
    sigs = {}
    for line in src.splitlines():
        s = line.strip()
        for nm in names:
            if re.search(r'\b(def|theorem|lemma|structure|class)\s+' + re.escape(nm) + r'\b', s):
                sigs.setdefault(nm, s)
    return sigs

print("--- Exemple guide : signatures extraites des sources astar_lean ---")
for mod, names in [
    ('Graph', ['WeightedGraph', 'pathCost']),
    ('Heuristic', ['Admissible', 'zero_admissible']),
    ('Optimality', ['IsTrueRemainingCost', 'admissible_implies_optimal']),
    ('Consistency', ['consistent_implies_path_bound', 'consistent_implies_f_monotone']),
]:
    sigs = extract_signatures(mod, names)
    for nm in names:
        print(f"  Astar/{mod}.lean :: {nm}")
        print(f"    {sigs.get(nm, '(non trouve -- verifier le nom)')}")
print("--- fin ---")


--- Exemple guide : signatures extraites des sources astar_lean ---
  Astar/Graph.lean :: WeightedGraph
    structure WeightedGraph (V : Type*) where
  Astar/Graph.lean :: pathCost
    def pathCost : Path V → NNReal
  Astar/Heuristic.lean :: Admissible
    def Admissible (h hStar : V → NNReal) : Prop :=
  Astar/Heuristic.lean :: zero_admissible
    theorem zero_admissible (hStar : V → NNReal) : Admissible (fun _ => (0 : NNReal)) hStar :=
  Astar/Optimality.lean :: IsTrueRemainingCost
    def IsTrueRemainingCost (hStar : V → NNReal) (goal : V) : Prop :=
  Astar/Optimality.lean :: admissible_implies_optimal
    theorem admissible_implies_optimal (h hStar : V → NNReal) (hAdm : Admissible h hStar)
  Astar/Consistency.lean :: consistent_implies_path_bound
    theorem consistent_implies_path_bound (h : V → NNReal) (goal : V)
  Astar/Consistency.lean :: consistent_implies_f_monotone
    theorem consistent_implies_f_monotone (h : V → NNReal)
--- fin ---


In [9]:
# Exercice 1 : un graphe pondere concret -- predisez puis verifiez un cout
#
# Objectif : on definit un WeightedGraph sur Fin 3. Predisez (de tête) le cout du
# chemin [0, 1, 2], puis DECOMMENTEZ le #eval pour verifier avec le lake.
# (TODO etudiant) : predisez, puis decommentez et executez (via run_lean).

snippet_ex1 = '''
import Mathlib
import Astar.Graph

open Astar

-- Graphe a 3 sommets (Fin 3). Arcs : 0->1 de poids 2, 1->2 de poids 3.
def G3 : WeightedGraph (Fin 3) := ⟨fun i j =>
    if i = 0 ∧ j = 1 then 2
    else if i = 1 ∧ j = 2 then 3
    else 0⟩

-- TODO etudiant : quel est le cout du chemin [0, 1, 2] ? (edge 0-1 + edge 1-2)
-- Decommentez pour verifier votre prédiction :
-- #eval pathCost G3 [0, 1, 2]
'''

print("--- Exercice 1 (squelette a completer) ---")
print(snippet_ex1)
print("--- fin ---")


--- Exercice 1 (squelette a completer) ---

import Mathlib
import Astar.Graph

open Astar

-- Graphe a 3 sommets (Fin 3). Arcs : 0->1 de poids 2, 1->2 de poids 3.
def G3 : WeightedGraph (Fin 3) := ⟨fun i j =>
    if i = 0 ∧ j = 1 then 2
    else if i = 1 ∧ j = 2 then 3
    else 0⟩

-- TODO etudiant : quel est le cout du chemin [0, 1, 2] ? (edge 0-1 + edge 1-2)
-- Decommentez pour verifier votre prédiction :
-- #eval pathCost G3 [0, 1, 2]

--- fin ---


In [10]:
# Exercice 2 : prouvez a la main que l'heuristique nulle est admissible
#
# Objectif : completer le `sorry` du `example` SANS utiliser le lemme
# `zero_admissible` du lake.
# Indice : en ℝ>=0, l'inegalite `0 ≤ hStar n` se ferme par `exact zero_le`.
# (TODO etudiant) : remplacez `sorry`, puis decommentez run_lean pour verifier.

snippet_ex2 = '''
import Mathlib
import Astar.Heuristic

open Astar

variable {V : Type*} (G : WeightedGraph V)

example (hStar : V → NNReal) : Admissible (fun _ => (0 : NNReal)) hStar := by
  intro n
  sorry   -- TODO etudiant : montrer 0 ≤ hStar n
'''

print("--- Exercice 2 (preuve a completer) ---")
print(snippet_ex2)
print("--- fin ---")


--- Exercice 2 (preuve a completer) ---

import Mathlib
import Astar.Heuristic

open Astar

variable {V : Type*} (G : WeightedGraph V)

example (hStar : V → NNReal) : Admissible (fun _ => (0 : NNReal)) hStar := by
  intro n
  sorry   -- TODO etudiant : montrer 0 ≤ hStar n

--- fin ---


In [11]:
# Exercice 3 : exhibez une heuristique NON admissible (contre-exemple)
#
# Objectif : construire une heuristique `h_bad` qui VIOLE `Admissible`.
# Indice : une constante `fun _ => 100` n'est admissible que si `hStar ≤ 100`
# partout ; choisissez un `hStar` qui prend une valeur > 100 quelque part et
# formalisez la réfutation.
# (TODO etudiant) : ecrivez l'example de non-admissibilite, puis decommentez.

snippet_ex3 = '''
import Mathlib
import Astar.Heuristic

open Astar

variable {V : Type*} (G : WeightedGraph V)

-- Heuristique trop optimiste partout.
def h_bad : V → NNReal := fun _ => 100

-- TODO etudiant : exhibez un hStar tel que ¬ Admissible h_bad hStar.
-- example (hStar : V → NNReal) (h : ∃ n, 100 < hStar n) :
--     ¬ Admissible h_bad hStar := by
--   sorry
'''

print("--- Exercice 3 (contre-exemple a construire) ---")
print(snippet_ex3)
print("--- fin ---")


--- Exercice 3 (contre-exemple a construire) ---

import Mathlib
import Astar.Heuristic

open Astar

variable {V : Type*} (G : WeightedGraph V)

-- Heuristique trop optimiste partout.
def h_bad : V → NNReal := fun _ => 100

-- TODO etudiant : exhibez un hStar tel que ¬ Admissible h_bad hStar.
-- example (hStar : V → NNReal) (h : ∃ n, 100 < hStar n) :
--     ¬ Admissible h_bad hStar := by
--   sorry

--- fin ---


## Conclusion

Ce notebook a visité le lake **`astar_lean`** (0 `sorry`, 4 modules), qui prouve
**formellement** l'optimalité de A\* sous heuristique admissible.

### Ce qui est prouvé

- **Modèle** (`Graph`) : graphe pondéré `ℝ≥0`, coût additif `pathCost` d'un chemin.
- **Prédicats** (`Heuristic`) : `Admissible` (globale) et `Consistent` (locale),
  avec leurs propriétés de base — dont la **connexion à Dijkstra** (`h ≡ 0`).
- **Optimalité** (`Optimality`) : le théorème phare `admissible_implies_optimal`
  — la borne en `f` qui garantit qu'A\* renvoie un chemin de coût optimal.
- **Téléscopage** (`Consistency`) : la consistance implique l'admissibilité (par
  récurrence le long du chemin) ET la monotonie de `f` (non-ré-expansion).

### La chaîne, honnêtement

`astar_lean` prouve la **forme abstraite** de l'optimalité (per #4048) : `hStar` est
une borne inférieure, le modèle isole le cœur mathématique (coût additif + borne en
`f`) sans modéliser la file de priorité complète. C'est délibéré — plus propre
pédagogiquement, et le mécanisme exact d'optimalité est bien là. La réalisabilité de
`hStar` (graphe fini ⇒ minimum atteint) reste abstraite.

### Où aller ensuite

- **Théorie** : Hart, Nilsson & Raphael (1968) ; Russell & Norvig, *AI: A Modern
  Approach* §3.5 (A\* et connexion à Dijkstra).
- **Lake** : [`astar_lean`](../../Search/astar_lean/) (README + sources `Astar/*.lean`).
- **Série** : les autres lakes #4038 (`sensitivity_lean`, `finiteness_lean`…) et
  leurs compagnons Lean-N.

**Navigation** : [<< Lean-17b Knots Invariants](Lean-17-Knots-b-Invariants-Companion.ipynb) | [Index](README.md)
